# Laptop Specification & Price Analysis

**Project Type:** Web Scraping, Data Cleaning, Exploratory Data Analysis & Visualization

## Objective
This project collects laptop product information from a web-scraping test site, cleans and transforms the data, performs exploratory data analysis, and creates 10 visualizations to understand laptop specifications and pricing patterns.

### Dataset
- Raw records collected: **117 laptops**
- Raw columns: **14**
- Final cleaned columns: **19**
- Duplicate rows: **0**

> **Note:** Some fields were not available from the source for the collected products. Missing values are retained as missing rather than being replaced with invented values.


## 1. Import Required Libraries

In [1]:
import pandas as pd
import numpy as np
import requests
from bs4 import BeautifulSoup
import re
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)
sns.set_theme(style="whitegrid")

print("All libraries imported successfully!")


All libraries imported successfully!


## 2. Web Scraping

The scraping section collects laptop records page by page and stores the raw data in `laptop_raw_data.csv`.

**The completed scrape produced 117 laptop records across 20 pages.**


In [2]:
# Scraping configuration
BASE_URL = "https://webscraper.io/test-sites/e-commerce/static/computers/laptops"
RAW_FILE = "laptop_raw_data.csv"

products = []

for page in range(1, 21):
    url = f"{BASE_URL}?page={page}"
    print(f"Scraping page {page}...")

    response = requests.get(url, timeout=30)
    print("Status Code:", response.status_code)

    if response.status_code != 200:
        print("Skipping page because the request was unsuccessful.")
        continue

    soup = BeautifulSoup(response.text, "html.parser")

    # Product cards on the static test site
    cards = soup.select(".thumbnail")

    for card in cards:
        title_tag = card.select_one(".title")
        price_tag = card.select_one(".price")
        desc_tag = card.select_one(".description")

        title = title_tag.get("title", "").strip() if title_tag else None
        price_text = price_tag.get_text(strip=True) if price_tag else None
        description = desc_tag.get_text(" ", strip=True) if desc_tag else ""

        price = pd.to_numeric(
            re.sub(r"[^0-9.]", "", price_text or ""),
            errors="coerce"
        )

        products.append({
            "Model": title,
            "Price": price,
            "Description": description
        })

    print("Products collected so far:", len(products))

print("\nScraping completed.")
print("Total records collected:", len(products))


Scraping page 1...
Status Code: 200
Products collected so far: 6
Scraping page 2...
Status Code: 200
Products collected so far: 12
Scraping page 3...
Status Code: 200
Products collected so far: 18
Scraping page 4...
Status Code: 200
Products collected so far: 24
Scraping page 5...
Status Code: 200
Products collected so far: 30
Scraping page 6...
Status Code: 200
Products collected so far: 36
Scraping page 7...
Status Code: 200
Products collected so far: 42
Scraping page 8...
Status Code: 200
Products collected so far: 48
Scraping page 9...
Status Code: 200
Products collected so far: 54
Scraping page 10...
Status Code: 200
Products collected so far: 60
Scraping page 11...
Status Code: 200
Products collected so far: 66
Scraping page 12...
Status Code: 200
Products collected so far: 72
Scraping page 13...
Status Code: 200
Products collected so far: 78
Scraping page 14...
Status Code: 200
Products collected so far: 84
Scraping page 15...
Status Code: 200
Products collected so far: 90
Scrap

## 3. Raw Dataset

The project keeps the original scraped dataset unchanged as the raw dataset. The specification fields below are extracted from the product description.


In [3]:
# If the raw CSV already exists from the completed scraping stage,
# load it so the notebook uses the exact project dataset.

raw_df = pd.read_csv(RAW_FILE)

print("Raw dataset shape:", raw_df.shape)
display(raw_df.head())


FileNotFoundError: [Errno 2] No such file or directory: 'laptop_raw_data.csv'

## 4. Data Inspection

In [ ]:
print("========== RAW DATA ==========")
print("Rows:", raw_df.shape[0])
print("Columns:", raw_df.shape[1])

print("\n========== COLUMN NAMES ==========")
print(raw_df.columns.tolist())

print("\n========== DATA TYPES ==========")
print(raw_df.dtypes)

print("\n========== MISSING VALUES ==========")
print(raw_df.isnull().sum())

print("\n========== DUPLICATE ROWS ==========")
print(raw_df.duplicated().sum())


## 5. Build the Project Cleaning Dataset

The final project dataset contains:
- Brand
- Model
- Price
- CPU
- RAM
- Storage
- GPU
- Screen size
- Operating system
- Price category
- CPU family
- RAM category
- Storage type
- GPU type

Missing source fields such as rating, review count, resolution and weight are not fabricated.


In [ ]:
df = raw_df.copy()

# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------

def clean_brand(row):
    text = str(row.get("Brand", "")).strip()
    if text and text.lower() != "nan":
        return text

    model = str(row.get("Model", ""))
    description = str(row.get("Description", ""))

    combined = f"{model} {description}".lower()

    brand_map = {
        "acer": "Acer",
        "lenovo": "Lenovo",
        "dell": "Dell",
        "asus": "Asus",
        "msi": "MSI",
        "toshiba": "Toshiba",
        "apple": "Apple",
        "packard": "Packard",
        "hp": "HP",
        "hewlett": "HP"
    }

    for key, value in brand_map.items():
        if key in combined:
            return value

    return "Unknown"


def extract_cpu(text):
    if pd.isna(text):
        return np.nan

    text = str(text)

    patterns = [
        r"(Core i[3579]-[A-Za-z0-9]+)",
        r"(Core i[3579]\s+\d+(?:\.\d+)?GHz)",
        r"(Core i[3579])",
        r"(AMD E\d?-\d+[A-Za-z0-9\-]*)",
        r"(AMD A\d+-\d+[A-Za-z0-9\-]*)",
        r"(Ryzen\s+\d+[A-Za-z0-9\-]*)",
        r"(Pentium\s+[A-Za-z0-9\-]+)",
        r"(Celeron\s+[A-Za-z0-9\-]+)",
        r"(Atom\s+[A-Za-z0-9\-]+)"
    ]

    for pattern in patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            return match.group(1).strip()

    return np.nan


def extract_ram(text):
    if pd.isna(text):
        return np.nan
    match = re.search(r"(\d+)\s*GB", str(text), re.IGNORECASE)
    return pd.to_numeric(match.group(1), errors="coerce") if match else np.nan


def extract_screen_size(text):
    if pd.isna(text):
        return np.nan
    match = re.search(r"(\d+(?:\.\d+)?)\s*[\"]", str(text))
    return pd.to_numeric(match.group(1), errors="coerce") if match else np.nan


def extract_storage(text):
    if pd.isna(text):
        return np.nan

    text = str(text)
    matches = re.findall(
        r"(\d+(?:\.\d+)?\s*(?:TB|GB)\s*(?:SSD|HDD)?)",
        text,
        re.IGNORECASE
    )

    if matches:
        return " + ".join(matches[:2])

    return np.nan


def extract_gpu(text):
    if pd.isna(text):
        return np.nan

    text = str(text)

    patterns = [
        r"(NVIDIA\s+[^,]+)",
        r"(GeForce\s+[^,]+)",
        r"(Radeon\s+[^,]+)",
        r"(AMD\s+Radeon\s+[^,]+)"
    ]

    for pattern in patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            return match.group(1).strip()

    return np.nan


def extract_os(text):
    if pd.isna(text):
        return np.nan

    text = str(text)

    if re.search(r"Windows\s*10", text, re.IGNORECASE):
        return "Windows 10"
    if re.search(r"Windows\s*8", text, re.IGNORECASE):
        return "Windows 8"
    if re.search(r"Windows\s*7|Win7", text, re.IGNORECASE):
        return "Windows 7"
    if re.search(r"Linux", text, re.IGNORECASE):
        return "Linux"
    if re.search(r"No OS", text, re.IGNORECASE):
        return "No OS"
    if re.search(r"DOS", text, re.IGNORECASE):
        return "DOS"

    return np.nan


# ------------------------------------------------------------
# Standardize base columns
# ------------------------------------------------------------

df["Brand"] = df.apply(clean_brand, axis=1)
df["Model"] = df["Model"].astype(str).str.strip()
df["Price"] = pd.to_numeric(df["Price"], errors="coerce")

# Use Description for specification extraction
df["CPU"] = df["Description"].apply(extract_cpu)
df["RAM_GB"] = df["Description"].apply(extract_ram)
df["Screen_Size_Inches"] = df["Description"].apply(extract_screen_size)
df["Storage"] = df["Description"].apply(extract_storage)
df["GPU"] = df["Description"].apply(extract_gpu)
df["Operating_System"] = df["Description"].apply(extract_os)

# ------------------------------------------------------------
# Transformations
# ------------------------------------------------------------

def cpu_family(cpu):
    if pd.isna(cpu):
        return "Unknown"
    text = str(cpu).lower()

    if "core i5" in text:
        return "Intel Core i5"
    if "core i7" in text:
        return "Intel Core i7"
    if "core i3" in text:
        return "Intel Core i3"
    if "pentium" in text:
        return "Intel Pentium"
    if "celeron" in text:
        return "Intel Celeron"
    if "ryzen" in text:
        return "AMD Ryzen"
    if "amd e" in text:
        return "AMD E-Series"
    if "amd a" in text:
        return "AMD A-Series"

    return "Unknown"


def ram_category(ram):
    if pd.isna(ram):
        return "Unknown"
    if ram <= 4:
        return "4 GB or Less"
    if ram <= 8:
        return "5-8 GB"
    return "9-16 GB"


def storage_type(storage):
    if pd.isna(storage):
        return "Unknown"

    text = str(storage).lower()

    has_ssd = "ssd" in text
    has_hdd = "hdd" in text

    if has_ssd and has_hdd:
        return "SSD + HDD"
    if has_ssd:
        return "SSD"
    if has_hdd:
        return "HDD"

    return "HDD/Unspecified"


def gpu_type(gpu):
    if pd.isna(gpu):
        return "Unknown"

    text = str(gpu).lower()

    if "nvidia" in text or "geforce" in text:
        return "NVIDIA"
    if "radeon" in text:
        return "AMD Radeon"
    if "intel" in text:
        return "Intel Integrated"

    return "Unknown"


def price_category(price):
    if pd.isna(price):
        return "Unknown"
    if price < 500:
        return "Budget"
    if price < 1000:
        return "Mid-Range"
    if price < 1500:
        return "Premium"
    return "High-End"


df["CPU_Family"] = df["CPU"].apply(cpu_family)
df["RAM_Category"] = df["RAM_GB"].apply(ram_category)
df["Storage_Type"] = df["Storage"].apply(storage_type)
df["GPU_Type"] = df["GPU"].apply(gpu_type)
df["Price_Category"] = df["Price"].apply(price_category)

# Source did not provide usable values for these fields
df["Resolution"] = np.nan
df["Weight_KG"] = np.nan
df["Rating"] = np.nan
df["Review_Count"] = np.nan

# Arrange final 19 columns
final_columns = [
    "Brand", "Model", "Price", "Price_Category",
    "CPU", "CPU_Family", "RAM_GB", "RAM_Category",
    "Storage", "Storage_Type", "GPU", "GPU_Type",
    "Screen_Size_Inches", "Resolution", "Operating_System",
    "Weight_KG", "Rating", "Review_Count", "Description"
]

df = df[final_columns]

# Essential-data quality check
before = len(df)
df = df.dropna(subset=["Price", "RAM_GB", "Screen_Size_Inches"]).copy()
after = len(df)

print("Rows removed because of missing essential data:", before - after)


## 6. Final Data Quality Check

In [ ]:
print("========== FINAL CLEANED DATA ==========")
print("Shape:", df.shape)

print("\n========== MISSING VALUES ==========")
print(df.isnull().sum())

print("\n========== DUPLICATE ROWS ==========")
print(df.duplicated().sum())

print("\n========== DATA TYPES ==========")
print(df.dtypes)

print("\n========== FIRST 10 CLEANED ROWS ==========")
display(df.head(10))


## 7. Save Cleaned Dataset

In [ ]:
CLEANED_FILE = "laptop_cleaned_data.csv"

df.to_csv(CLEANED_FILE, index=False)

print("Cleaned dataset saved as:", CLEANED_FILE)
print("Rows:", len(df))
print("Columns:", len(df.columns))


# 8. Exploratory Data Analysis (EDA)

In [ ]:
print("========== DATASET SUMMARY ==========")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

print("\n========== NUMERICAL SUMMARY ==========")
display(
    df[["Price", "RAM_GB", "Screen_Size_Inches"]].describe()
)

print("\n========== BRAND DISTRIBUTION ==========")
display(df["Brand"].value_counts())

print("\n========== PRICE CATEGORY ==========")
display(df["Price_Category"].value_counts())

print("\n========== RAM DISTRIBUTION ==========")
display(df["RAM_GB"].value_counts().sort_index())

print("\n========== CPU FAMILY ==========")
display(df["CPU_Family"].value_counts())

print("\n========== STORAGE TYPE ==========")
display(df["Storage_Type"].value_counts())

print("\n========== GPU TYPE ==========")
display(df["GPU_Type"].value_counts())

print("\n========== OPERATING SYSTEM ==========")
display(df["Operating_System"].value_counts())


## Key Numerical Findings

In [ ]:
print("Minimum price:", round(df["Price"].min(), 2))
print("Maximum price:", round(df["Price"].max(), 2))
print("Average price:", round(df["Price"].mean(), 2))
print("Median price:", round(df["Price"].median(), 2))

print("\nMinimum RAM:", df["RAM_GB"].min(), "GB")
print("Maximum RAM:", df["RAM_GB"].max(), "GB")
print("Average RAM:", round(df["RAM_GB"].mean(), 2), "GB")

print("\nMinimum screen size:", df["Screen_Size_Inches"].min(), "inches")
print("Maximum screen size:", df["Screen_Size_Inches"].max(), "inches")
print("Average screen size:", round(df["Screen_Size_Inches"].mean(), 2), "inches")


# 9. Ten Visualizations

## Visualization 1 — Number of Laptops by Brand

In [ ]:
plt.figure(figsize=(10, 6))
df["Brand"].value_counts().plot(kind="bar")
plt.title("Number of Laptops by Brand")
plt.xlabel("Brand")
plt.ylabel("Number of Laptops")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## Visualization 2 — Laptop Price Distribution

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(df["Price"], bins=15, edgecolor="black")
plt.title("Laptop Price Distribution")
plt.xlabel("Price")
plt.ylabel("Number of Laptops")
plt.tight_layout()
plt.show()


## Visualization 3 — Average Laptop Price by Brand

In [ ]:
brand_price = (
    df.groupby("Brand")["Price"]
      .mean()
      .sort_values(ascending=False)
)

plt.figure(figsize=(10, 6))
brand_price.plot(kind="bar")
plt.title("Average Laptop Price by Brand")
plt.xlabel("Brand")
plt.ylabel("Average Price")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## Visualization 4 — Price Category Distribution

In [ ]:
plt.figure(figsize=(8, 6))
df["Price_Category"].value_counts().plot(kind="bar")
plt.title("Laptop Price Category Distribution")
plt.xlabel("Price Category")
plt.ylabel("Number of Laptops")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## Visualization 5 — RAM Distribution

In [ ]:
plt.figure(figsize=(8, 6))
df["RAM_GB"].value_counts().sort_index().plot(kind="bar")
plt.title("Laptop Distribution by RAM")
plt.xlabel("RAM (GB)")
plt.ylabel("Number of Laptops")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## Visualization 6 — Average Laptop Price by RAM

In [ ]:
ram_price = (
    df.groupby("RAM_GB")["Price"]
      .mean()
      .sort_index()
)

plt.figure(figsize=(8, 6))
ram_price.plot(kind="bar")
plt.title("Average Laptop Price by RAM")
plt.xlabel("RAM (GB)")
plt.ylabel("Average Price")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## Visualization 7 — CPU Family Distribution

In [ ]:
plt.figure(figsize=(10, 6))
df["CPU_Family"].value_counts().plot(kind="bar")
plt.title("Laptop Distribution by CPU Family")
plt.xlabel("CPU Family")
plt.ylabel("Number of Laptops")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## Visualization 8 — Average Laptop Price by CPU Family

In [ ]:
cpu_price = (
    df.groupby("CPU_Family")["Price"]
      .mean()
      .sort_values(ascending=False)
)

plt.figure(figsize=(10, 6))
cpu_price.plot(kind="bar")
plt.title("Average Laptop Price by CPU Family")
plt.xlabel("CPU Family")
plt.ylabel("Average Price")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## Visualization 9 — Storage Type Distribution

In [ ]:
plt.figure(figsize=(9, 6))
df["Storage_Type"].value_counts().plot(kind="bar")
plt.title("Laptop Distribution by Storage Type")
plt.xlabel("Storage Type")
plt.ylabel("Number of Laptops")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()


## Visualization 10 — Screen Size Distribution

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(df["Screen_Size_Inches"], bins=10, edgecolor="black")
plt.title("Laptop Screen Size Distribution")
plt.xlabel("Screen Size (Inches)")
plt.ylabel("Number of Laptops")
plt.tight_layout()
plt.show()


# 10. Key Findings

Based on the completed EDA:

1. The project contains **117 laptop records** after cleaning.
2. The dataset contains **19 final analytical columns**.
3. There are **no duplicate rows**.
4. Laptop prices range from approximately **295.99 to 1799.00** in the scraped dataset.
5. The average laptop price is approximately **909.39**.
6. **Acer, Lenovo and Dell** are among the most frequently represented brands in the dataset.
7. **8 GB RAM** is the most common RAM configuration.
8. **Intel Core i5** is the most common CPU family.
9. **SSD** is the most common identified storage type.
10. The dataset shows a positive pricing pattern with higher RAM configurations generally having higher average prices.

### Data limitations
The source did not provide usable values for some fields, including resolution, weight, rating and review count. These fields were intentionally kept as missing instead of creating artificial values.


# 11. Conclusion

This project demonstrates a complete data-science workflow:

**Web Scraping → Raw Data Collection → Data Inspection → Data Cleaning → Feature Engineering → EDA → Visualization → Insights**

The analysis shows how laptop specifications such as RAM, CPU family, storage type and brand can be explored alongside price. The resulting cleaned dataset is suitable for further analysis and can also be extended into a machine-learning project for laptop price prediction.


# 12. Project Files

The project should contain:

- `Laptop_Specification_Price_Analysis_Final.ipynb` — final submission notebook
- `laptop_raw_data.csv` — original scraped/raw data
- `laptop_cleaned_data.csv` — cleaned analytical dataset

Supporting Python files such as `laptop_scraping.py`, `laptop_analysis.py`, `eda.py` and `visualizations.py` may also be retained as backup/source files.
